# **NYC Airbnb — Data Cleaning Project**

## **Project Overview**

This project focuses on cleaning and preprocessing a dataset containing Airbnb listings in New York City. The dataset includes information about property locations, neighborhoods, geographical coordinates, prices, room types, and availability.

The main goal is to improve data quality by identifying and correcting missing, inconsistent, and potentially inaccurate values.

### **Tasks**

1) Handle Missing Values — Identify missing data and determine appropriate methods for handling it.

2) Clean Categorical Data — Standardize neighborhood names and correct inconsistent categorical values.

3) Restore Missing Location Information — Fill missing neighborhood and neighborhood group values using available information.

4) Process Geographical Data — Examine latitude and longitude coordinates to identify missing, incorrect, or inconsistent locations.

5) Correct Data Types — Ensure that columns contain values in appropriate formats.

6) Validate Data Consistency — Identify and resolve inconsistencies between related columns.


## **What was done:**

The NYC Airbnb dataset was cleaned and preprocessed using Python and Pandas, with a focus on handling missing values and improving geographical data consistency.

**1. Missing Values Analysis**

Used Pandas methods such as isna() and sum() to identify and count missing values.

Examined missing values in neighbourhood and neighbourhood_group to determine how they could be restored.

**2. Restoring Missing Neighbourhood Groups**

Analyzed the relationship between neighbourhood and neighbourhood_group.

Created a mapping between neighborhoods and their corresponding boroughs using existing records.

Used Pandas mapping and missing-value handling techniques to fill missing neighbourhood_group values while preserving existing data.

Checked the remaining missing values after applying the mapping.

**3. Geographical Data Processing**

Worked with latitude and longitude columns to investigate ways of recovering missing neighborhood information using KNN in scikit-learn.

Explored the relationship between geographical coordinates and neighborhood locations.

**4. Data Validation**

Checked the dataset for remaining missing values.

Verified the results of the neighborhood group restoration process.



In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("Airbnb_Open_Data.csv")

In [ ]:
df.head(5)

In [ ]:
# delete duplicates|

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
# change columns names

In [ ]:
df.columns

In [ ]:
column_names = df.columns

In [ ]:
type(column_names)

In [ ]:
column_names = column_names.str.lower()

In [ ]:
column_names = column_names.str.strip()

In [ ]:
column_names = column_names.str.replace(" ", "_")

In [ ]:
column_names

In [ ]:
df.columns =column_names

In [ ]:
# чек host_id - все файно

In [ ]:
df['host_id'].dtype

In [ ]:
(df['host_id']<0).sum()

In [ ]:
# чек host_identity_verified - все добре

In [ ]:
df['host_identity_verified'].value_counts(dropna=False)

In [ ]:
df['host_name'].value_counts()

In [ ]:
# у ствопцю Name знайдемо рядки з символами окрім звичайних літер

In [ ]:
# create a mask

In [ ]:
invalid_host_names_mask = df['host_name'].str.contains(r"[^A-Za-z]", na=False)

In [ ]:
invalid_host_names_mask.sum()

In [ ]:
df.loc[
    invalid_host_names_mask
]

In [ ]:
# не думаю що ім'я треба трогати. Залишаємо як є

In [ ]:
# check neighbourhood_group

In [ ]:
df['neighbourhood_group'].value_counts(dropna=False)

In [ ]:
valid_neighbourhood_group_map = {
    "manhatan":"Manhattan",
    "brookln":"Brooklyn"
}

In [ ]:
df['neighbourhood_group'] = df['neighbourhood_group'].replace(valid_neighbourhood_group_map)

In [ ]:
# Washington Heights - Manhattan
# Clinton Hill - Brooklyn
# Williamsburg - Brooklyn
# East Village - Manhattan
# Bushwick - Brooklyn
# Harlem - Manhattan
# Prospect Heights - Brooklyn
# East Harlem - Manhattan
# Woodside - Queens
# Upper East Side - Manhattan
# Chelsea - Manhattan
# Eastchester - Bronx
# Chinatown - Manhattan
# Queens Village - Queens
# Bedford-Stuyvesant - Brooklyn
# Upper West Side - Manhattan

In [ ]:
df[df['neighbourhood_group'].isna()]

In [ ]:
# create a map

In [ ]:
neighbourhood_map = {
    "Washington Heights": "Manhattan",
    "Clinton Hill": "Brooklyn",
    "Williamsburg": "Brooklyn",
    "East Village": "Manhattan",
    "Bushwick": "Brooklyn",
    "Harlem": "Manhattan",
    "Prospect Heights": "Brooklyn",
    "East Harlem": "Manhattan",
    "Woodside": "Queens",
    "Upper East Side": "Manhattan",
    "Chelsea": "Manhattan",
    "Eastchester": "Bronx",
    "Chinatown": "Manhattan",
    "Queens Village": "Queens",
    "Bedford-Stuyvesant": "Brooklyn",
    "Upper West Side": "Manhattan"
}

In [ ]:
df['neighbourhood_group'] = df['neighbourhood_group'].fillna(df['neighbourhood'].map(neighbourhood_map))

In [ ]:
df['neighbourhood'].isna().sum()

In [ ]:
#lets fill the neighbourhood using KNeighborsClassifier:

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
# rows where neighbourhood and the coordinates are known

In [ ]:
known = df.dropna(subset = ["neighbourhood","lat",'long'])

In [ ]:
X_train = known[["lat",'long']]

In [ ]:
y_train = known["neighbourhood"]

In [ ]:
# train a model

In [ ]:
model = KNeighborsClassifier(n_neighbors=5, weights="distance")

In [ ]:
model.fit(X_train, y_train)

In [ ]:
# now lets create a "test" set. By that I mean rows where we lat and long but not neighbourhood

In [ ]:
missing_neighbourhood_mask = (
    df['neighbourhood'].isna() &
    df['lat'].notna() &
    df['long'].notna()
)

In [ ]:
# lets look at them

In [ ]:
df.loc[
    missing_neighbourhood_mask, 'neighbourhood'
] = model.predict(df.loc[missing_neighbourhood_mask,["lat","long"]])

In [ ]:
df["neighbourhood_group"].isna().sum()

In [ ]:
#

In [ ]:
df['lat'].isna().sum()

In [ ]:
df[df['lat'].isna()]

In [ ]:
df['long'].dtype

In [ ]:
len(list(column_names))

In [ ]:
df['country'].value_counts(dropna=False)

In [ ]:
#Lets fill the missing country based on lat and long

In [ ]:
df['country'] = df['country'].fillna('United States')

In [ ]:
# country - fixed

In [ ]:
df.columns

In [ ]:
# country_code - valid

In [ ]:
df['country_code'].value_counts(dropna=False)

In [ ]:
df['country_code'].fillna("US", inplace=True)

In [ ]:
# instant_bookable - valid

In [ ]:
df['instant_bookable'].value_counts()

In [ ]:
# cancellation_policy - valid

In [ ]:
df['cancellation_policy'].value_counts()

In [ ]:
# check other variables

In [ ]:
df['construction_year'].notna()

In [ ]:
df['construction_year'] = df.loc[
    df['construction_year'].notna(), 'construction_year'
].astype("Int64")

In [ ]:
df['price'].dtype

In [ ]:
df['valid_price'] = df['price']

In [ ]:
#lets fix it

In [ ]:
df['valid_price'] = df['valid_price'].str.replace(r"[^0-9]","",regex=True)

In [ ]:
df['valid_price']

In [ ]:
df['valid_price'] = df['valid_price'].astype("Float64")

In [ ]:
df['valid_price'].mean()

In [ ]:
#service_fee

In [ ]:
df['service_fee']
df.loc[
    df['service_fee'].notna(),'service_fee'
] = df['service_fee'].str.replace(r"[^0-9]","",regex=True)

In [ ]:
df['service_fee']
df.loc[
    df['service_fee'].notna(),'service_fee'
] = df['service_fee'].str.strip()

In [ ]:
df['service_fee'] = df['service_fee'].astype("Float64")

In [ ]:
df['service_fee'].mean()

In [ ]:
#minimum_nights
import numpy as np


In [ ]:
df['minimum_nights'] = np.abs(df['minimum_nights'])

In [ ]:
df['minimum_nights'] = df['minimum_nights'].astype("Int64")

In [ ]:
df['minimum_nights'].max()

In [ ]:
# minimum_nights - has big outliers

In [ ]:
df.columns

In [ ]:
df['number_of_reviews'].dtype

In [ ]:
df['price'] = df['valid_price']

In [ ]:
df.drop(columns=["valid_price"], inplace=True)

In [ ]:
df.info()

In [ ]:
df['last_review'].to_Datetime()

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
file_path = "/content/drive/MyDrive/Python Cleaning Datasets/cleaned_dataset.csv"

df.to_csv(file_path, index=False)

In [ ]:
# продовженя 10_09

In [ ]:
df = pd.read_csv("cleaned_dataset.csv")

/tmp/ipykernel_491/3693996082.py:1: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("cleaned_dataset.csv")


In [ ]:
df.isna().sum()

,0
id,0
name,250
host_id,0
host_identity_verified,289
host_name,404
neighbourhood_group,0
neighbourhood,0
lat,8
long,8
country,0


In [ ]:
df['number_of_reviews'].isna().sum()

np.int64(183)

In [ ]:
#instant_bookable

In [ ]:
df['cancellation_policy'].value_counts()

,count
cancellation_policy,
moderate,34162
strict,33929
flexible,33891


In [ ]:
# last_review

In [ ]:
import pandas as pd

In [ ]:
df['last_review']

dtype('O')

In [ ]:
df['last_review'] = pd.to_datetime(df['last_review'],errors='coerce')

In [ ]:
df['last_review'].dtype

dtype('<M8[ns]')

In [ ]:
# reviews_per_month

In [ ]:
df['reviews_per_month'].dtype

dtype('float64')

In [ ]:
df['reviews_per_month'].value_counts(dropna=False)

,count
reviews_per_month,
NaN,15818
0.03,1665
0.05,1489
1.00,1454
0.09,1264
...,...
47.11,1
10.52,1
10.98,1


In [ ]:
df['reviews_per_month_B'] = df['number_of_reviews']/12

In [ ]:
df[['reviews_per_month_B','reviews_per_month']].head()

,reviews_per_month_B,reviews_per_month
0,0.75,0.21
1,3.75,0.38
2,0.00,NaN
3,22.50,4.64
4,0.75,0.10


In [ ]:
df.head()

,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,...,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules,license,reviews_per_month_B
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,...,10.0,9.0,2021-10-19,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...,NaN,0.75
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,...,30.0,45.0,2022-05-21,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...,NaN,3.75
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,NaN,Elise,Manhattan,Harlem,40.80902,-73.94190,United States,...,3.0,0.0,NaT,NaN,5.0,1.0,352.0,"I encourage you to use my kitchen, cooking and...",NaN,0.00
3,1002755,NaN,85098326012,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,...,30.0,270.0,2019-07-05,4.64,4.0,1.0,322.0,NaN,NaN,22.50
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,verified,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,United States,...,10.0,9.0,2018-11-19,0.10,3.0,1.0,289.0,"Please no smoking in the house, porch or on th...",NaN,0.75


In [ ]:
#calculated_host_listings_count

In [ ]:
df['calculated_host_listings_count'].argmax()

np.int64(57424)

In [ ]:
df.iloc[[57424]]

,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,...,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules,license,reviews_per_month_B
57424,32716611,Smart Tribeca Studio w/ Great views + Indoor p...,76431087156,unconfirmed,Blueground,Manhattan,Tribeca,40.715794,-74.005875,United States,...,30.0,1.0,2022-01-12,0.57,5.0,332.0,328.0,NaN,NaN,0.083333
